# SENTINEL-BC experiments

This notebook uses the application's modules and saved experiment. It never builds a second pipeline or silently refits. Run `python train.py --mode full` first. Research measurements only, never clinical clearance.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display, Image
from sentinel_bc.config import ROOT, load_config
from sentinel_bc.data import load_data, partitions
from sentinel_bc.inference import Engine
from sentinel_bc.explainability import local_table
from sentinel_bc.safety_gate import SafetyPolicy, decide
x, y = load_data()
roles = partitions(y, load_config())
assert sum(y == 1) == 212
assert len(np.unique(np.concatenate(list(roles.values())))) == 569
pd.DataFrame([{"role": k, "n": len(v), "malignant": int(y[v].sum())} for k, v in roles.items()])

## Fitting-only data inspection
Correlated morphology measurements can share attribution. Units follow the source values; no invented physical-unit conversion is applied.

In [ ]:
x.iloc[roles['fit']].describe().round(3)

## The frozen experiment
CV selected the primary estimator before any final test metrics. The table below reads cached test results.

In [ ]:
directory = ROOT / 'results/full'
metrics = json.loads((directory / 'metrics.json').read_text())
display(pd.DataFrame(metrics['models']).T[['accuracy', 'malignant_recall', 'fn', 'brier']])
metrics['selective']

In [ ]:
display(Image(filename=str(directory / 'figures/calibration.png')))
display(Image(filename=str(directory / 'figures/safety.png')))

## Labeled development example
Inference uses the same version-checked engine as Gradio. The source diagnosis is educational context and is never an input feature.

In [ ]:
engine = Engine()
examples = pd.read_csv(directory / 'demo_cases.csv')
case = examples.loc[:0, engine.features]
print(examples.loc[0, ['example_name', 'source_label', 'source_row']].to_dict())
prediction = engine.predict(case)[0]
prediction

In [ ]:
display(local_table(engine, case).head(10))

## Fail-closed behavior and monotonicity
An impossible negative radius is out of scope. Holding support and uncertainty fixed, increasing malignant probability cannot relax referral.

In [ ]:
invalid = case.copy()
invalid.iloc[0, 0] = -1
assert engine.predict(invalid)[0]['decision'] == 'OUT_OF_SCOPE_INPUT'
u = {'entropy': .1, 'disagreement': .01, 'mc_std': .01}
flags = [decide(float(p), u, [True, False], True, False, engine.policy)[0] != 'LOW_RISK_ESTIMATE' for p in np.linspace(0,1,101)]
assert np.all(np.diff(np.asarray(flags, dtype=int)) >= 0)
print('Invalid input rejected; monotone referral verified.')

## Retraining
Use `python train.py --mode quick` for an installation check and `python train.py --mode full --output-root experiments/repeat` for a separate repeat. Do not tune using the locked test. A new scientific comparison needs a new evaluation plan.

## Interpretation
Always report accepted accuracy **with coverage**. Zero observed accepted malignancies in a small sample does not establish zero real-world risk. Calibration worsened the current held-out Brier/NLL. Conformal coverage is marginal under exchangeability, not individual clinical protection.